# WiDS Wildfire Structural-Gate Survival Ensemble

In [1]:
import os, json, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, GradientBoostingClassifier

SEED = 20260423
random.seed(SEED)
np.random.seed(SEED)

HORIZONS = [12, 24, 48]
HIT_GATE_M = 5000.0
CV_FULL_ALPHA = 0.65
SEEDS = (104729, 104759, 104761, 104773, 104779)

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUTPUT_PATH = os.path.join(WORK_DIR, "submission.csv")


def find_data_dir():
    needed = {"train.csv", "test.csv", "sample_submission.csv"}
    candidates = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldatathon26",
        "/kaggle/input/widsworldwide-global-datathon-2026",
        ".",
        "/mnt/data",
    ]
    for path in candidates:
        if os.path.isdir(path) and needed.issubset(set(os.listdir(path))):
            return path
    for root0 in ["/kaggle/input", "/mnt/data", "."]:
        if os.path.isdir(root0):
            for root, _, files in os.walk(root0):
                if needed.issubset(set(files)):
                    return root
    raise FileNotFoundError("train.csv, test.csv, sample_submission.csv not found")


DATA_DIR = find_data_dir()
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

target_time = train_df["time_to_hit_hours"].to_numpy(dtype=float)
target_event = train_df["event"].to_numpy(dtype=int)


def s(df, col, default=0.0):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").astype(float).replace([np.inf, -np.inf], np.nan).fillna(default)
    return pd.Series(default, index=df.index, dtype=float)


def engineer(df):
    r = df.copy()
    dist = s(r, "dist_min_ci_0_5h").clip(lower=1.0)
    area_first = s(r, "area_first_ha").clip(lower=0.0)
    area_growth_abs = s(r, "area_growth_abs_0_5h").clip(lower=0.0)
    perims = s(r, "num_perimeters_0_5h").clip(lower=1.0)
    dt = s(r, "dt_first_last_0_5h").clip(lower=0.0)
    closing = s(r, "closing_speed_m_per_h")
    radial_rate = s(r, "radial_growth_rate_m_per_h")
    centroid_speed = s(r, "centroid_speed_m_per_h")
    align_abs = s(r, "alignment_abs").clip(lower=0.0, upper=1.0)

    dist_km = dist / 1000.0
    radius_first = np.sqrt(area_first * 10000.0 / np.pi)
    area_final = (area_first + area_growth_abs).clip(lower=0.0)
    radius_final = np.sqrt(area_final * 10000.0 / np.pi)
    closing_pos = closing.clip(lower=0.0)
    radial_pos = radial_rate.clip(lower=0.0)
    effective_speed = closing_pos + radial_pos

    r["dist_km"] = dist_km
    r["log_dist"] = np.log1p(dist)
    r["inv_dist_km"] = 1.0 / (dist_km + 0.1)
    r["near5k"] = (dist < HIT_GATE_M).astype(int)
    r["dist_to_5k_km"] = (dist - HIT_GATE_M) / 1000.0
    r["radius_m"] = radius_first
    r["radius_final_m"] = radius_final
    r["margin_m"] = dist - HIT_GATE_M - radius_first
    r["margin_final_m"] = dist - HIT_GATE_M - radius_final
    r["radius_to_dist"] = radius_first / (dist + 1.0)
    r["area_per_dist"] = area_first / (dist_km + 0.1)
    r["log_area_per_dist"] = np.log1p(area_first) - np.log1p(dist)
    r["positive_closing"] = closing_pos
    r["positive_radial"] = radial_pos
    r["effective_speed"] = effective_speed
    r["eta_eff"] = np.where(effective_speed > 0.01, dist / effective_speed, 9999.0)
    r["log_eta_eff"] = np.log1p(np.clip(r["eta_eff"], 0.0, 9999.0))
    r["has_motion"] = (perims > 1).astype(int)
    r["log_num_perim"] = np.log1p(perims)
    r["motion_strength"] = r["has_motion"] * (np.log1p(dt) + np.log1p(perims))
    r["speed_align"] = centroid_speed * align_abs
    r["close_align"] = closing_pos * (0.2 + align_abs)
    r["radial_align"] = radial_pos * (0.2 + align_abs)
    r["small_area"] = (area_first < 10.0).astype(int)
    r["tiny_area"] = (area_first < 5.0).astype(int)
    r["large_area"] = (area_first > 500.0).astype(int)

    for col, period in [("event_start_hour", 24), ("event_start_dayofweek", 7), ("event_start_month", 12)]:
        v = s(r, col)
        r[f"{col}_sin"] = np.sin(2.0 * np.pi * v / period)
        r[f"{col}_cos"] = np.cos(2.0 * np.pi * v / period)

    month = s(r, "event_start_month")
    hour = s(r, "event_start_hour")
    r["peak_season"] = month.isin([7, 8, 9]).astype(int)
    r["month_7"] = (month == 7).astype(int)
    r["month_8"] = (month == 8).astype(int)
    r["month_9"] = (month == 9).astype(int)
    r["hour_16_19"] = hour.between(16, 19).astype(int)
    r["night_0_5"] = hour.between(0, 5).astype(int)
    r["no_motion_small_afternoon_jul"] = (
        (perims == 1)
        & (area_first < 10.0)
        & hour.between(16, 18)
        & month.isin([4, 7])
    ).astype(int)

    r = r.replace([np.inf, -np.inf], 0.0).fillna(0.0)
    return r


train_fe = engineer(train_df)
test_fe = engineer(test_df)

event_dist_max = train_df.loc[target_event == 1, "dist_min_ci_0_5h"].max()
nonevent_dist_min = train_df.loc[target_event == 0, "dist_min_ci_0_5h"].min()
if np.isfinite(event_dist_max) and np.isfinite(nonevent_dist_min) and event_dist_max < nonevent_dist_min:
    HIT_GATE_M = 5000.0 if (event_dist_max < 5000.0 < nonevent_dist_min) else 0.5 * (event_dist_max + nonevent_dist_min)
    train_fe = engineer(train_df)
    test_fe = engineer(test_df)

feature_cols = [
    "num_perimeters_0_5h",
    "dt_first_last_0_5h",
    "low_temporal_resolution_0_5h",
    "area_first_ha",
    "log1p_area_first",
    "dist_min_ci_0_5h",
    "dist_km",
    "log_dist",
    "radius_m",
    "margin_m",
    "area_per_dist",
    "alignment_abs",
    "event_start_hour",
    "event_start_dayofweek",
    "event_start_month",
    "event_start_hour_sin",
    "event_start_hour_cos",
    "event_start_dayofweek_sin",
    "event_start_dayofweek_cos",
    "event_start_month_sin",
    "event_start_month_cos",
    "peak_season",
    "month_7",
    "month_8",
    "month_9",
    "hour_16_19",
    "night_0_5",
    "has_motion",
    "log_num_perim",
    "motion_strength",
    "tiny_area",
    "small_area",
    "large_area",
    "no_motion_small_afternoon_jul",
]
feature_cols = [c for c in feature_cols if c in train_fe.columns and c in test_fe.columns]

near_train_mask = train_df["dist_min_ci_0_5h"].to_numpy(dtype=float) < HIT_GATE_M
near_test_mask = test_df["dist_min_ci_0_5h"].to_numpy(dtype=float) < HIT_GATE_M
near_train_idx = np.where(near_train_mask)[0]
near_test_idx = np.where(near_test_mask)[0]

X_near = train_fe.loc[near_train_idx, feature_cols].reset_index(drop=True)
X_test_near = test_fe.loc[near_test_idx, feature_cols].reset_index(drop=True)
time_near = target_time[near_train_idx]
log_time_near = np.log1p(time_near)
y_near = {h: (time_near <= h).astype(int) for h in HORIZONS}

conditional_prior = {h: float(np.mean(y_near[h])) for h in HORIZONS}


def positive_proba(model, X):
    if len(X) == 0:
        return np.zeros(0, dtype=float)
    p = model.predict_proba(X)
    classes = getattr(model, "classes_", np.array([0, 1]))
    if p.shape[1] == 1:
        return np.ones(len(X), dtype=float) if int(classes[0]) == 1 else np.zeros(len(X), dtype=float)
    pos = int(np.where(classes == 1)[0][0])
    return p[:, pos]


def classifier_candidate(builder, seeds=SEEDS):
    oof_by_h, test_by_h = {}, {}
    for h in HORIZONS:
        y = y_near[h]
        if len(np.unique(y)) < 2:
            oof_by_h[h] = np.full(len(y), float(y[0]))
            test_by_h[h] = np.full(len(X_test_near), float(y[0]))
            continue

        min_class = int(np.bincount(y, minlength=2).min())
        n_splits = max(2, min(5, min_class))
        oof = np.zeros(len(y), dtype=float)
        oof_count = np.zeros(len(y), dtype=float)
        test_cv = np.zeros(len(X_test_near), dtype=float)
        test_cv_count = 0
        test_full = np.zeros(len(X_test_near), dtype=float)

        for seed in seeds:
            cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
            for tr_idx, va_idx in cv.split(X_near, y):
                model = builder(seed)
                model.fit(X_near.iloc[tr_idx], y[tr_idx])
                oof[va_idx] += positive_proba(model, X_near.iloc[va_idx])
                oof_count[va_idx] += 1.0
                if len(X_test_near):
                    test_cv += positive_proba(model, X_test_near)
                    test_cv_count += 1

            model = builder(seed)
            model.fit(X_near, y)
            if len(X_test_near):
                test_full += positive_proba(model, X_test_near)

        oof = oof / np.maximum(oof_count, 1.0)
        if len(X_test_near):
            test_full /= len(seeds)
            test_cv = test_cv / max(test_cv_count, 1)
            test_pred = CV_FULL_ALPHA * test_full + (1.0 - CV_FULL_ALPHA) * test_cv
        else:
            test_pred = np.zeros(0, dtype=float)

        oof_by_h[h] = np.clip(oof, 0.0, 1.0)
        test_by_h[h] = np.clip(test_pred, 0.0, 1.0)

    return oof_by_h, test_by_h


def regression_candidate(builder, seeds=SEEDS):
    n = len(X_near)
    oof_mu = np.zeros(n, dtype=float)
    oof_count = np.zeros(n, dtype=float)
    test_mu_cv = np.zeros(len(X_test_near), dtype=float)
    test_mu_cv_count = 0
    test_mu_full = np.zeros(len(X_test_near), dtype=float)

    for seed in seeds:
        cv = KFold(n_splits=5, shuffle=True, random_state=seed)
        for tr_idx, va_idx in cv.split(X_near):
            model = builder(seed)
            model.fit(X_near.iloc[tr_idx], log_time_near[tr_idx])
            oof_mu[va_idx] += model.predict(X_near.iloc[va_idx])
            oof_count[va_idx] += 1.0
            if len(X_test_near):
                test_mu_cv += model.predict(X_test_near)
                test_mu_cv_count += 1

        model = builder(seed)
        model.fit(X_near, log_time_near)
        if len(X_test_near):
            test_mu_full += model.predict(X_test_near)

    oof_mu = oof_mu / np.maximum(oof_count, 1.0)
    residuals = log_time_near - oof_mu

    if len(X_test_near):
        test_mu_full /= len(seeds)
        test_mu_cv /= max(test_mu_cv_count, 1)
        test_mu = CV_FULL_ALPHA * test_mu_full + (1.0 - CV_FULL_ALPHA) * test_mu_cv
    else:
        test_mu = np.zeros(0, dtype=float)

    oof_by_h, test_by_h = {}, {}
    for h in HORIZONS:
        th = math.log1p(h)
        oof_prob = np.zeros(n, dtype=float)
        for i in range(n):
            rr = np.delete(residuals, i) if n > 1 else residuals
            oof_prob[i] = np.mean(rr <= th - oof_mu[i])
        test_prob = np.array([np.mean(residuals <= th - mu) for mu in test_mu], dtype=float)
        oof_by_h[h] = np.clip(oof_prob, 0.0, 1.0)
        test_by_h[h] = np.clip(test_prob, 0.0, 1.0)

    return oof_by_h, test_by_h


def build_logit(seed):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(C=0.3, solver="lbfgs", max_iter=5000)),
    ])


def build_et_classifier(seed):
    return ExtraTreesClassifier(
        n_estimators=180,
        min_samples_leaf=2,
        max_features="sqrt",
        bootstrap=False,
        random_state=seed,
        n_jobs=1,
    )


def build_gbc(seed):
    return GradientBoostingClassifier(
        n_estimators=90,
        learning_rate=0.04,
        max_depth=2,
        subsample=0.85,
        random_state=seed,
    )


def build_ridge(seed):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=10.0)),
    ])


def build_et_regressor(seed):
    return ExtraTreesRegressor(
        n_estimators=180,
        min_samples_leaf=2,
        max_features="sqrt",
        bootstrap=False,
        random_state=seed,
        n_jobs=1,
    )


candidates_oof = {}
candidates_test = {}

for name, builder in [
    ("logit", build_logit),
    ("et_clf", build_et_classifier),
    ("gbc", build_gbc),
]:
    oof, tst = classifier_candidate(builder)
    candidates_oof[name] = oof
    candidates_test[name] = tst

for name, builder in [
    ("ridge_reg", build_ridge),
    ("et_reg", build_et_regressor),
]:
    oof, tst = regression_candidate(builder)
    candidates_oof[name] = oof
    candidates_test[name] = tst

BLEND_WEIGHTS = {
    12: {"logit": 0.0695, "et_clf": 0.1390, "gbc": 0.2440, "ridge_reg": 0.0655, "et_reg": 0.4820},
    24: {"logit": 0.0370, "et_clf": 0.0480, "gbc": 0.3805, "ridge_reg": 0.1435, "et_reg": 0.3910},
    48: {"logit": 0.0240, "et_clf": 0.3730, "gbc": 0.2895, "ridge_reg": 0.1565, "et_reg": 0.1570},
}


def blend_candidate_dict(cdict, h, n_rows):
    weights = BLEND_WEIGHTS[h]
    out = np.zeros(n_rows, dtype=float)
    total = 0.0
    for name, w in weights.items():
        if name in cdict:
            out += w * cdict[name][h]
            total += w
    if total <= 0:
        return np.full(n_rows, conditional_prior[h], dtype=float)
    return np.clip(out / total, 0.0, 1.0)


near_oof_pred = {h: blend_candidate_dict(candidates_oof, h, len(X_near)) for h in HORIZONS}
near_test_pred = {h: blend_candidate_dict(candidates_test, h, len(X_test_near)) for h in HORIZONS}

oof_pred = np.zeros((len(train_df), 4), dtype=float)
test_pred = np.zeros((len(test_df), 4), dtype=float)

for j, h in enumerate(HORIZONS):
    oof_pred[near_train_idx, j] = near_oof_pred[h]
    test_pred[near_test_idx, j] = near_test_pred[h]

oof_pred[:, 3] = 1.0
test_pred[:, 3] = 1.0

oof_pred = np.maximum.accumulate(np.clip(oof_pred, 0.0, 1.0), axis=1)
test_pred = np.maximum.accumulate(np.clip(test_pred, 0.0, 1.0), axis=1)
oof_pred[:, 3] = 1.0
test_pred[:, 3] = 1.0


def c_index(time, event, risk):
    conc = 0.0
    comp = 0.0
    n = len(time)
    for i in range(n):
        if event[i] != 1:
            continue
        for j in range(n):
            if i == j or time[i] >= time[j]:
                continue
            comp += 1.0
            if risk[i] > risk[j]:
                conc += 1.0
            elif risk[i] == risk[j]:
                conc += 0.5
    return conc / comp if comp else 0.5


def brier_at(time, event, prob, horizon):
    valid = ~((event == 0) & (time < horizon))
    y = ((event == 1) & (time <= horizon)).astype(float)[valid]
    return float(np.mean((np.clip(prob[valid], 0, 1) - y) ** 2))


risk_oof = 0.3 * oof_pred[:, 1] + 0.4 * oof_pred[:, 2] + 0.3 * oof_pred[:, 3]
oof_c = c_index(target_time, target_event, risk_oof)
oof_b24 = brier_at(target_time, target_event, oof_pred[:, 1], 24)
oof_b48 = brier_at(target_time, target_event, oof_pred[:, 2], 48)
oof_b72 = brier_at(target_time, target_event, oof_pred[:, 3], 72)
oof_wb = 0.3 * oof_b24 + 0.4 * oof_b48 + 0.3 * oof_b72
oof_hybrid = 0.3 * oof_c + 0.7 * (1.0 - oof_wb)

submission = pd.DataFrame({
    "event_id": test_df["event_id"].to_numpy(),
    "prob_12h": test_pred[:, 0],
    "prob_24h": test_pred[:, 1],
    "prob_48h": test_pred[:, 2],
    "prob_72h": test_pred[:, 3],
})
submission = sample_sub[["event_id"]].merge(submission, on="event_id", how="left", validate="one_to_one")

required = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
assert list(submission.columns) == required
assert len(submission) == len(sample_sub)
assert submission["event_id"].equals(sample_sub["event_id"])
assert submission["event_id"].nunique() == len(submission)
vals = submission[required[1:]].to_numpy(dtype=float)
assert np.isfinite(vals).all()
assert ((vals >= 0.0) & (vals <= 1.0)).all()
assert np.all(vals[:, 0] <= vals[:, 1] + 1e-12)
assert np.all(vals[:, 1] <= vals[:, 2] + 1e-12)
assert np.all(vals[:, 2] <= vals[:, 3] + 1e-12)

os.makedirs(WORK_DIR, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

diagnostics = {
    "data_dir": DATA_DIR,
    "hit_gate_m": float(HIT_GATE_M),
    "near_train": int(near_train_mask.sum()),
    "near_test": int(near_test_mask.sum()),
    "conditional_prior": conditional_prior,
    "oof_hybrid_proxy": float(oof_hybrid),
    "oof_c_index_proxy": float(oof_c),
    "oof_brier_24": float(oof_b24),
    "oof_brier_48": float(oof_b48),
    "oof_brier_72": float(oof_b72),
}
with open(os.path.join(WORK_DIR, "diagnostics.json"), "w") as f:
    json.dump(diagnostics, f, indent=2)

print("DATA_DIR:", DATA_DIR)
print("OOF hybrid proxy:", round(oof_hybrid, 6), "| C:", round(oof_c, 6), "| WB:", round(oof_wb, 6))
print("Near train/test:", int(near_train_mask.sum()), int(near_test_mask.sum()), "| gate:", HIT_GATE_M)
print("Saved:", OUTPUT_PATH)
print(submission.head())

DATA_DIR: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
OOF hybrid proxy: 0.976587 | C: 0.937479 | WB: 0.006652
Near train/test: 69 28 | gate: 5000.0
Saved: /kaggle/working/submission.csv
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.000000  0.000000  0.000000       1.0
1  13353600  0.590045  0.942149  0.990318       1.0
2  13942327  0.000000  0.000000  0.000000       1.0
3  16112781  0.782160  0.950688  0.992264       1.0
4  17132808  0.000000  0.000000  0.000000       1.0
